In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_mensajeria = config['MENSAJERIA_OLTP']
    config_etl = config['ETL_PROCESS']

# Construct the database URL
url_mensajeria = (f"{config_mensajeria['drivername']}://{config_mensajeria['user']}:{config_mensajeria['password']}@{config_mensajeria['host']}:"
          f"{config_mensajeria['port']}/{config_mensajeria['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_mensajeria)
etl_conn = create_engine(url_etl)

In [3]:
fechas = pd.read_sql_table('mensajeria_estadosservicio',mensajeria)
estados_servicios = pd.read_sql_table('mensajeria_estado',mensajeria)
servicio = pd.read_sql_table('mensajeria_servicio',mensajeria)

fechas = fechas.drop(columns=['foto','observaciones','es_prueba','foto_binary'])

fechas['day_of_week'] = fechas['fecha'].dt.weekday
fechas['year'] = fechas['fecha'].dt.year
fechas['month'] = fechas['fecha'].dt.month

# fechas['Estado'] = mensajeria_estado['nombre']
estados_servicios.loc[0,"nombre"] = "Recogido en origen"
estados_servicios.loc[3,"nombre"] = "Cerrado"

fechas = fechas.merge(
    estados_servicios[['id','nombre']],
    left_on ='estado_id',
    right_on='id',
    how='left'
)
fechas.rename(columns={'nombre':'estado'},inplace=True)

fechas.drop(columns=['id_y'],inplace=True)

servicio.drop(columns=['descripcion', 'nombre_solicitante', 'fecha_solicitud',
       'hora_solicitud', 'fecha_deseada', 'hora_deseada', 'nombre_recibe',
       'telefono_recibe', 'descripcion_pago', 'ida_y_regreso', 'activo',
       'novedades', 'cliente_id', 'destino_id', 'mensajero_id', 'origen_id',
       'tipo_pago_id', 'tipo_servicio_id', 'tipo_vehiculo_id', 'usuario_id',
       'prioridad', 'ciudad_destino_id', 'ciudad_origen_id',
       'hora_visto_por_mensajero', 'visto_por_mensajero',
       'descripcion_multiples_origenes', 'mensajero2_id', 'mensajero3_id',
       'multiples_origenes', 'asignar_mensajero', 'es_prueba',
       'descripcion_cancelado'],inplace=True)

fechas

servicio = servicio.merge(
    fechas,
    left_on='id',
    right_on='servicio_id',
    how='left'
)

servicio.drop(columns=['estado_id'],inplace=True)

servicio["fecha_hora"] = pd.to_datetime(
    servicio["fecha"].dt.date.astype(str) +
    " " +
    servicio["hora"].astype(str),
    errors="coerce"
)

servicio

,id,id_x,fecha,hora,servicio_id,day_of_week,year,month,estado,fecha_hora
0,34,123,2023-10-26,09:46:03,34,3,2023,10,Iniciado,2023-10-26 09:46:03
1,35,124,2023-10-26,11:18:14,35,3,2023,10,Iniciado,2023-10-26 11:18:14
2,35,125,2023-10-28,14:43:08,35,5,2023,10,Con mensajero Asignado,2023-10-28 14:43:08
3,35,126,2023-10-28,19:45:18,35,5,2023,10,Recogido en origen,2023-10-28 19:45:18
4,35,195,2023-12-07,01:25:34,35,3,2023,12,Entregado en destino,2023-12-07 01:25:34
...,...,...,...,...,...,...,...,...,...,...
128397,28437,128615,2024-08-31,10:44:39,28437,5,2024,8,Con mensajero Asignado,2024-08-31 10:44:39
128398,28437,128609,2024-08-31,10:40:33,28437,5,2024,8,Iniciado,2024-08-31 10:40:33
128399,28437,128652,2024-08-31,11:07:02,28437,5,2024,8,Recogido en origen,2024-08-31 11:07:02
128400,28437,128675,2024-08-31,11:27:20,28437,5,2024,8,Entregado en destino,2024-08-31 11:27:20


In [4]:
dim_fechahora = (
    servicio[["fecha_hora"]]
    .drop_duplicates()
    .sort_values("fecha_hora")
    .reset_index(drop=True)
)

dim_fechahora["fecha_hora_key"] = (
    dim_fechahora.index + 1
)

dim_fechahora["year"] = dim_fechahora["fecha_hora"].dt.year
dim_fechahora["month"] = dim_fechahora["fecha_hora"].dt.month
dim_fechahora["day"] = dim_fechahora["fecha_hora"].dt.day
dim_fechahora["hour"] = dim_fechahora["fecha_hora"].dt.hour
dim_fechahora["minute"] = dim_fechahora["fecha_hora"].dt.minute
dim_fechahora["day_of_week"] = dim_fechahora["fecha_hora"].dt.day_name()

dim_fechahora

,fecha_hora,fecha_hora_key,year,month,day,hour,minute,day_of_week
0,2023-09-19 16:22:18,1,2023.0,9.0,19.0,16.0,22.0,Tuesday
1,2023-09-19 16:30:05,2,2023.0,9.0,19.0,16.0,30.0,Tuesday
2,2023-09-19 16:35:52,3,2023.0,9.0,19.0,16.0,35.0,Tuesday
3,2023-09-19 16:37:54,4,2023.0,9.0,19.0,16.0,37.0,Tuesday
4,2023-09-19 16:49:17,5,2023.0,9.0,19.0,16.0,49.0,Tuesday
...,...,...,...,...,...,...,...,...
127006,2024-08-31 15:03:42,127007,2024.0,8.0,31.0,15.0,3.0,Saturday
127007,2024-08-31 15:04:26,127008,2024.0,8.0,31.0,15.0,4.0,Saturday
127008,2024-08-31 15:13:47,127009,2024.0,8.0,31.0,15.0,13.0,Saturday
127009,2024-08-31 15:22:35,127010,2024.0,8.0,31.0,15.0,22.0,Saturday


In [5]:
trans_servicio = servicio.merge(
    dim_fechahora[['fecha_hora_key','fecha_hora']],
    on='fecha_hora',
    how='left'
)

trans_servicio.drop(columns=['id_x','fecha','hora','servicio_id','day_of_week','year','month'],inplace=True)



In [6]:
tiempos = trans_servicio.pivot_table(
    index='id',
    columns='estado',
    values='fecha_hora',
    aggfunc='first'
).reset_index()

tiempos['minutos_asignacion'] = (
    tiempos['Con mensajero Asignado']
    - tiempos['Iniciado']
).dt.total_seconds() / 60

tiempos['minutos_recogida'] = (
    tiempos['Recogido en origen']
    - tiempos['Con mensajero Asignado']
).dt.total_seconds() / 60

tiempos['minutos_entrega'] = (
    tiempos['Entregado en destino']
    - tiempos['Recogido en origen']
).dt.total_seconds() / 60

tiempos['minutos_cierre'] = (
    tiempos['Cerrado']
    - tiempos['Entregado en destino']
).dt.total_seconds() / 60



In [7]:
fact = trans_servicio.pivot_table(
    index='id',
    columns='estado',
    values='fecha_hora_key',
    aggfunc='first'
).reset_index()


fact = fact.merge(
    tiempos[
        [
            'id',
            'minutos_asignacion',
            'minutos_recogida',
            'minutos_entrega',
            'minutos_cierre'
        ]
    ],
    on='id',
    how='left'
)

fact.columns

fact = fact.rename(columns={
    'Solicitado': 'fk_fecha_solicitado',
    'Con mensajero Asignado': 'fk_fecha_asignado',
    'Recogido en origen': 'fk_fecha_recogido',
    'Cerrado': 'fk_fecha_entregado'
})

fact

estado,id,fk_fecha_entregado,fk_fecha_asignado,Con novedad,Entregado en destino,Iniciado,fk_fecha_recogido,minutos_asignacion,minutos_recogida,minutos_entrega,minutos_cierre
0,7,100.0,83.0,NaN,101.0,1.0,127011.0,34649.033333,NaN,NaN,-291.916667
1,8,NaN,207.0,NaN,33623.0,2.0,5238.0,132704.633333,80359.583333,79234.283333,NaN
2,9,NaN,239.0,NaN,NaN,2.0,NaN,144182.933333,NaN,NaN,NaN
3,10,NaN,241.0,NaN,17280.0,3.0,6715.0,144177.250000,73728.666667,30816.666667,NaN
4,11,NaN,178.0,NaN,NaN,4.0,1468.0,116436.083333,76155.933333,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
28425,28464,NaN,126982.0,126985.0,126992.0,126972.0,126990.0,24.716667,25.433333,19.216667,NaN
28426,28465,NaN,126991.0,NaN,127010.0,126988.0,127001.0,10.166667,52.400000,48.766667,NaN
28427,28466,NaN,126995.0,127005.0,NaN,126994.0,127008.0,7.250000,53.433333,NaN,NaN
28428,28467,NaN,127000.0,127004.0,NaN,126996.0,NaN,16.883333,NaN,NaN,NaN
